# Day 43 — Conditional Probability in Machine Learning

This notebook covers Conditional Probability ($P(A \mid B)$), Shrunken Sample Spaces, Base Rate Fallacy, and Pandas Data Filtering

## Validation goals
- Understand conditional probability $P(A \mid B)$ as a method for updating likelihoods with new evidence
- Explore how conditioning shrinks and normalizes the universal sample space down to localized subsets
- Analyze real-world pitfalls like the Base Rate Fallacy in medical diagnostics and anomaly detection
- Implement conditional probability calculations programmatically using Pandas data filtering (`df[df[...] == 1]`)

Conditional probability is the mathematical foundation behind how machine learning classifiers make predictions, how spam filters catch junk emails, and how diagnostic tools evaluate medical test results. Let’s break it down from first principles.

### 1. What is Conditional Probability?

### Concept Explanation

In everyday life, our certainty about an event changes the moment we receive new information.
- Unconditional Probability ($P(A)$): The likelihood of an event happening out of nowhere, with zero context. (Example: What is the chance it will rain today in general?)
- Conditional Probability ($P(A \vert{} B)$): The likelihood of an event ($A$) happening given that another event or piece of evidence ($B$) has already occurred.

Read the notation $P(A \vert{} B)$ as: "The probability of $A$, given $B$."

**Relatable Example: The Umbrella**

Without looking outside, the general probability that you will need an umbrella today might be low ($P(\text{Umbrella}) = 0.10$).
- But then you look out the window and see dark, heavy storm clouds. The event "storm clouds" is your new evidence ($B$).
- The conditional probability of needing an umbrella given that you see storm clouds shoots up drastically ($P(\text{Umbrella} \vert{} \text{Storm Clouds}) = 0.90$).

Machine learning models do this constantly: they update their predictions as new data features roll in.

### 2. The Mathematical Formula

To calculate conditional probability mathematically, we use the intersection of events (where both $A$ and $B$ happen together) divided by the probability of the given condition ($B$):
$$P(A \vert{} B) = \frac{P(A \text{ and } B)}{P(B)}$$

* **$P(A \text{ and } B)$**: The probability that both events happen together.
* **$P(B)$**: The probability of the conditioning event happening at all.

Note: This only works when $P(B)$ is greater than zero (you can't condition on an impossible event!).

### 3. Relatable ML Example: Email Spam Filtering

Imagine you are building a spam classifier for an inbox. You want to know if an incoming email is spam ($A$) based on whether it contains the word "Winner" ($B$).
- Event $A$: The email is spam.
- Event $B$: The email contains the word "Winner".

When your model calculates $P(\text{Spam} \vert{} \text{Contains 'Winner'})$, it is asking: "Out of all the emails that contain the word 'Winner', what percentage of them turned out to be actual spam?" If that conditional probability is $0.95$ (95%), the model flags the email and routes it straight to the spam folder.

In [1]:
import pandas as pd

# Create a sample dataset of online store visitors
data = {
    'visited_pricing_page': [1, 0, 1, 1, 0, 0, 1, 0, 1, 1],
    'made_purchase': [1, 0, 0, 1, 0, 0, 1, 0, 0, 1]
}

df = pd.DataFrame(data)

print("--- Dataset Sample ---")
print(df.head(), "\n")

# Question: What is the probability of making a purchase, given that the user visited the pricing page?
# Step 1: Filter rows where the user visited the pricing page (Event B)
pricing_visitors = df[df['visited_pricing_page'] == 1]

# Step 2: Out of those, calculate how many made a purchase (Event A and B)
purchased_after_pricing = pricing_visitors[pricing_visitors['made_purchase'] == 1]

# Step 3: Compute conditional probability
cond_prob = len(purchased_after_pricing) / len(pricing_visitors)

print(f"Total visitors who checked pricing: {len(pricing_visitors)}")
print(f"Visitors who checked pricing AND purchased: {len(purchased_after_pricing)}")
print(f"P(Purchase | Visited Pricing): {cond_prob:.2f}")

--- Dataset Sample ---
   visited_pricing_page  made_purchase
0                     1              1
1                     0              0
2                     1              0
3                     1              1
4                     0              0 

Total visitors who checked pricing: 6
Visitors who checked pricing AND purchased: 4
P(Purchase | Visited Pricing): 0.67


### Guided Question:

In your own words, explain how adding a "given condition" ($B$) changes the sample space compared to finding a regular, unconditional probability ($A$).

### Answer:

At its core, adding a "given condition" ($B$) acts like a filter or a zoom lens that fundamentally resizes the universe you are looking at.

1. The Unconditional View: Looking at the Whole World

When you calculate a regular, unconditional probability like $P(A)$, your sample space is the entire universal set of all possible outcomes ($\Omega$).
- The Question: "Across everything that could possibly happen, what fraction belongs to $A$?"
- The Denominator: The total size of the entire sample space ($100\%$ or $1.0$).
- Analogy: Pulling a random card out of a brand-new, shuffled 52-card deck. Your sample space is all 52 cards.

2. The Conditional View: Shrinking the Sample Space

When you introduce a condition—written as $P(A \vert B)$—you are told: "Assume event $B$ has already happened.
- "This single piece of information changes the rules of reality for your calculation:Discard the Irrelevant: You immediately throw away every single outcome in the sample space where $B$ is false. The universe outside of $B$ no longer exists for your problem.
- Redefine the Baseline: Your sample space shrinks from the global universe ($\Omega$) down to a smaller, localized sub-universe ($B$).
- Re-evaluate the Target: Within this new, shrunken world of $B$, you now ask: "What portion of this restricted space is occupied by $A$?"

**Analogy**: Sticking with the card deck, suppose someone looks at the deck and tells you: "The card you drew is a face card." Your sample space instantly shrinks from 52 cards down to just the 12 face cards. If you want to know the probability that it's a King, your denominator is no longer 52; it is now 12.

3. How the Formula Reflects This Shift

This physical shrinking of the sample space is precisely why the conditional probability formula is structured the way it is:
**$$P(A \vert B) = \frac{P(A \text{ and } B)}{P(B)}$$
- The Denominator ($P(B)$): Represents the size of your new, shrunken sample space. You are dividing everything by the probability of the condition itself, which normalizes the new universe so it sums back up to $1.0$.
- The Numerator ($P(A \text{ and } B)$): Represents the only part of $A$ that actually matters now—the overlap where $A$ and $B$ coexist inside that shrunken universe.

In software terms, unconditional probability is like querying an entire database table (`SELECT COUNT(A) / COUNT(*) FROM records`), whereas conditional probability is like applying a `WHERE` clause first (`SELECT COUNT(A AND B) / COUNT(B) FROM records WHERE B = true`).

### Guided Question:

If a medical test for a rare disease has a high conditional probability of coming back positive given that a patient actually has the disease, why is it still important to consider the base rate of the disease itself?

### Answer:

Even if a medical test has an exceptionally high conditional probability of detecting a disease when it is present—known as sensitivity or $P(\text{Positive Test} \vert{} \text{Disease})$—ignoring the base rate leads to severe statistical errors.The core reason is a mathematical phenomenon called the Base Rate Fallacy: Scale defeats accuracy when the target condition is extremely rare.

**The Anatomy of the Problem: Signal vs. Noise**

When a test is deployed on a population, it faces two entirely different groups:
- The Sick Group (The Signal): A tiny fraction of the population governed by the base rate ($P(\text{Disease})$).
- The Healthy Group (The Noise): The vast majority of the population ($1 - P(\text{Disease})$).

Even if a test has a tiny false-positive rate (meaning it misidentifies healthy people as sick only 1% of the time), applying that 1% error rate across a massive pool of healthy people creates a flood of false positives that can completely overwhelm the true signals from the sick people.

**A Concrete Numerical Example**

Imagine a rare disease that affects 1 in 10,000 people in a city of 1,000,000 people.

You use a state-of-the-art diagnostic test with 99% sensitivity (it correctly flags 99% of sick people) and a 99% specificity (meaning it has a 1% false-positive rate for healthy people).

Let's break down what happens when you test the entire city of 1,000,000:

- Who actually has the disease?Base rate: $\frac{1}{10,000} \times 1,000,000 = \mathbf{100\text{ people}}$ are truly sick.

- Who is truly healthy?

    - $1,000,000 - 100 = \mathbf{999,900\text{ people}}$ are healthy.

- How many True Positives do you get?

    - $99\%$ of the 100 sick people test positive = $99$ true positives.

- How many False Positives do you get?

    - $1\%$ false-positive rate applied to 999,900 healthy people = $9,999$ false positives.

Now, look at the total pool of positive test results generated by your app:
$$\text{Total Positives} = 99 \text{ (true)} + 9,999 \text{ (false)} = \mathbf{10,098\text{ positive tests}}$$

If a patient walks in and gets a positive test, what is the actual probability they have the disease ($P(\text{Disease} \vert{} \text{Positive Test})$)?$$P(\text{Disease} \vert{} \text{Positive Test}) = \frac{\text{True Positives}}{\text{Total Positives}} = \frac{99}{10,098} \approx \mathbf{0.0098 \text{ (less than } 1\% \text{!)}}$$

Despite the test being "99% accurate," a positive result means there is still a greater than 99% chance the patient is actually healthy.

Why the Base Rate is the Ultimate ModeratorIn Bayes' Theorem, the base rate acts as the prior probability—the baseline weight of reality before any evidence is gathered:$$P(\text{Disease} \vert{} \text{Positive Test}) = \frac{P(\text{Positive Test} \vert{} \text{Disease}) \times \mathbf{P(\text{Disease})}}{P(\text{Positive Test})}$$

If the base rate $P(\text{Disease})$ is microscopic, the numerator shrinks drastically.

In software engineering terms, this is identical to a low signal-to-noise ratio in logging. If you are monitoring a massive server farm processing billions of healthy requests ($99.999\%$ uptime), and an alert mechanism triggers a "false alarm" 0.01% of the time due to minor network blips, your monitoring dashboard will be flooded with thousands of fake alerts every day. Without factoring in the ultra-low base rate of actual server crashes, your system will constantly chase ghosts.

### Guided Question:

In the Python example above, how would you modify the script to calculate the reverse conditional probability: the probability that a user visited the pricing page, given that they made a purchase?

### Answer:

To calculate the reverse conditional probability—$P(\text{Visited Pricing} \mid \text{Made Purchase})$—you invert the steps so that making a purchase becomes the primary filtering condition (the new sample space).

Here is the updated step-by-step logic:
- Filter by the new condition: Filter the dataset to isolate only users who made a purchase (made_purchase == 1).
- Count the target overlap: Out of those purchasers, count how many visited the pricing page (visited_pricing_page == 1).
- Compute the ratio: Divide the number of purchasers who visited the pricing page by the total number of purchasers.

In [2]:
import pandas as pd

# Create a sample dataset of online store visitors
data = {
    'visited_pricing_page': [1, 0, 1, 1, 0, 0, 1, 0, 1, 1],
    'made_purchase': [1, 0, 0, 1, 0, 0, 1, 0, 0, 1]
}

df = pd.DataFrame(data)

# Reverse Question: P(Visited Pricing | Made Purchase)

# Step 1: Filter rows where the user made a purchase (New Event B)
purchasers = df[df['made_purchase'] == 1]

# Step 2: Out of those, calculate how many visited the pricing page (Event A and B)
pricing_and_purchased = purchasers[purchasers['visited_pricing_page'] == 1]

# Step 3: Compute reverse conditional probability
reverse_cond_prob = len(pricing_and_purchased) / len(purchasers)
# Calculates the mean of 'visited_pricing_page' restricted only to rows where 'made_purchase' is 1
# reverse_cond_prob = df[df['made_purchase'] == 1]['visited_pricing_page'].mean()

print(f"Total buyers who purchased: {len(purchasers)}")
print(f"Buyers who purchased AND checked pricing: {len(pricing_and_purchased)}")
print(f"P(Visited Pricing | Made Purchase): {reverse_cond_prob:.2f}")

Total buyers who purchased: 4
Buyers who purchased AND checked pricing: 4
P(Visited Pricing | Made Purchase): 1.00


**Understanding the Difference in Output**

- $P(\text{Purchase} \mid \text{Visited Pricing}) = \frac{4}{6} \approx 0.67$: Out of 6 people who checked the pricing page, 4 bought a product (measuring conversion intent of pricing page visitors).
- $P(\text{Visited Pricing} \mid \text{Purchase}) = \frac{4}{4} = 1.00$: Out of the 4 total buyers in this dataset, all 4 of them visited the pricing page before buying (measuring funnel necessity or path attribution).